# NLP Project 3 — Icelandic ASR
## Person 3: Whisper Small Fine-Tuning

**Dataset:** Samrómur 21.05 (OpenSLR 112)  
**Model:** OpenAI Whisper Small Multilingual  
**Platform:** Google Colab  
**Deadline:** November 23, 2026

### Objectives
1. Fine-tune Whisper Small on approximately 15 hours of Icelandic speech.
2. Monitor training and validation performance.
3. Record training hyperparameters and GPU usage.
4. Select the best checkpoint using validation results.
5. Generate predictions on the shared 3-hour test set.
6. Export the model, training logs, and predictions.

### Experimental Rules
- Use only the official training subset for fine-tuning.
- Use validation data for model selection.
- Keep the test set untouched until final evaluation.
- Record all hyperparameters and random seeds.
- Preserve Icelandic characters and original transcripts.
- Save checkpoints to persistent storage.

In [ ]:
# Reproducibility setup — Person 3
# Install the exact package versions documented in the repository.
# Run this cell in a fresh Colab environment before running other code.

%pip install -q -r https://raw.githubusercontent.com/RanimAlyaseen/icelandic-asr-project3/person3-whisper-reproducibility/training/requirements-person3.txt

In [ ]:
import sys
import platform
import subprocess
import torch
import pandas as pd

print("=" * 60)
print("NLP PROJECT 3 — TRAINING ENVIRONMENT")
print("=" * 60)

print(f"\nPython version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Operating system: {platform.system()}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory = gpu_properties.total_memory / (1024**3)

    print(f"\nGPU: {gpu_name}")
    print(f"GPU memory: {gpu_memory:.2f} GB")

    print("\nNVIDIA GPU information:")

    subprocess.run(["nvidia-smi"])

else:
    print("\nWARNING: No GPU detected!")
    print("Enable GPU from Runtime > Change runtime type.")

print("\nEnvironment inspection completed.")

NLP PROJECT 3 — TRAINING ENVIRONMENT

Python version: 3.13.16
PyTorch version: 2.14.1+cu130
Operating system: Linux
CUDA available: True

GPU: Tesla T4
GPU memory: 14.56 GB

NVIDIA GPU information:

Environment inspection completed.


## Step 2 — Dependencies and Model Initialization

### Objective
Install and verify the libraries required for Whisper Small fine-tuning.

### Base Model
- Model ID: `openai/whisper-small`
- Architecture: Encoder-decoder Transformer
- Parameters: Approximately 244 million
- Language: Icelandic (`is`)
- Task: Automatic Speech Recognition

### Required Libraries
- Transformers: Model and processor
- Datasets: Audio dataset handling
- Accelerate: GPU training
- Evaluate: WER and CER
- JiWER: Speech recognition evaluation
- SoundFile: Audio loading
- Librosa: Audio processing

### Validation Checks
1. Verify installed library versions.
2. Load the Whisper processor.
3. Load the pretrained model.
4. Configure Icelandic transcription.
5. Verify GPU compatibility.

In [ ]:
%pip install -q "evaluate==0.4.6" "jiwer==4.0.0"

In [ ]:
%pip install -q "evaluate==0.4.6" "jiwer==4.0.0"


## Step 2B — Whisper Small Model Initialization

### Objective
Load the pretrained Whisper Small Multilingual model and configure it for Icelandic automatic speech recognition.

### Model Configuration
- Model: openai/whisper-small
- Architecture: Encoder-decoder Transformer
- Language: Icelandic
- Task: Transcription

### Validation
- Verify model and processor loading.
- Check the total number of parameters.
- Verify Icelandic language configuration.
- Confirm the model is ready for inference and fine-tuning.

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-small"

print("=" * 60)
print("WHISPER SMALL — MODEL INITIALIZATION")
print("=" * 60)

# Load processor for Icelandic transcription
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="icelandic",
    task="transcribe"
)

print("\nProcessor loaded successfully.")

# Load pretrained model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID
)

print("Model loaded successfully.")

# Configure transcription language
model.generation_config.language = "icelandic"
model.generation_config.task = "transcribe"

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("\nMODEL INFORMATION")
print("-" * 60)

print(f"Model ID: {MODEL_ID}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params / 1e6:.2f}M")

print("\nLANGUAGE CONFIGURATION")
print("-" * 60)

print("Language:", model.generation_config.language)
print("Task:", model.generation_config.task)

print("\nDEVICE INFORMATION")
print("-" * 60)

print("Model device:", next(model.parameters()).device)
print("GPU available:", torch.cuda.is_available())

print("\nMODEL INITIALIZATION COMPLETE")

WHISPER SMALL — MODEL INITIALIZATION


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Model loaded successfully.

MODEL INFORMATION
------------------------------------------------------------
Model ID: openai/whisper-small
Total parameters: 241,734,912
Trainable parameters: 241,734,912
Model size: 241.73M

LANGUAGE CONFIGURATION
------------------------------------------------------------
Language: icelandic
Task: transcribe

DEVICE INFORMATION
------------------------------------------------------------
Model device: cpu
GPU available: True

MODEL INITIALIZATION COMPLETE


## Step 3 — Fine-Tuning Configuration

### Objective
Configure Whisper Small Multilingual for Icelandic ASR fine-tuning.

### Training Strategy
- Base model: `openai/whisper-small`
- Training data: Approximately 15 hours
- Validation data: Approximately 2 hours
- Test data: Approximately 3 hours
- GPU: NVIDIA Tesla T4
- Training method: Full fine-tuning
- Precision: FP16
- Gradient accumulation: Enabled
- Gradient checkpointing: Enabled if required

### Experimental Requirements
1. Record all hyperparameters.
2. Use a fixed random seed for reproducibility.
3. Monitor training and validation losses.
4. Calculate validation WER and CER.
5. Save checkpoints and training logs.
6. Select the best checkpoint using validation results.

### Important
The test dataset must not be used during training or checkpoint selection.

In [ ]:
import json
import random
import numpy as np
import torch

from pathlib import Path
from datetime import datetime, timezone

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Project configuration
TRAINING_CONFIG = {
    "project": "Icelandic ASR - NLP Project 3",
    "model_id": "openai/whisper-small",
    "language": "icelandic",
    "task": "transcribe",

    # Dataset targets
    "train_hours_target": 15,
    "validation_hours_target": 2,
    "test_hours_target": 3,

    # Initial training hyperparameters
    "learning_rate": 1e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "linear",

    # Memory optimization
    "fp16": True,
    "gradient_checkpointing": True,

    # Experiment tracking
    "seed": SEED,
    "logging_steps": 25,
    "save_strategy": "epoch",
    "evaluation_strategy": "epoch",
    "save_total_limit": 2,

    # Model selection
    "metric_for_best_model": "wer",
    "greater_is_better": False,

    # Audio
    "target_sample_rate": 16000,
    "max_audio_duration_seconds": 30,

    # Metadata
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),
    "pytorch_version": torch.__version__
}

# Effective batch size for one GPU
effective_batch_size = (
    TRAINING_CONFIG["per_device_train_batch_size"]
    * TRAINING_CONFIG["gradient_accumulation_steps"]
)

# Save configuration locally
output_dir = Path("/content/icelandic_asr")
output_dir.mkdir(parents=True, exist_ok=True)

config_path = output_dir / "training_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(TRAINING_CONFIG, f, indent=4, ensure_ascii=False)

print("=" * 60)
print("WHISPER SMALL — FINE-TUNING CONFIGURATION")
print("=" * 60)

print(f"\nModel: {TRAINING_CONFIG['model_id']}")
print(f"GPU: {TRAINING_CONFIG['gpu']}")
print(f"Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"Batch size: {TRAINING_CONFIG['per_device_train_batch_size']}")
print(f"Gradient accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"Effective batch size: {effective_batch_size}")
print(f"FP16: {TRAINING_CONFIG['fp16']}")
print(f"Gradient checkpointing: {TRAINING_CONFIG['gradient_checkpointing']}")

print(f"\nConfiguration saved to: {config_path}")
print("\nSTEP 3 COMPLETE")

WHISPER SMALL — FINE-TUNING CONFIGURATION

Model: openai/whisper-small
GPU: CPU
Learning rate: 1e-05
Epochs: 3
Batch size: 2
Gradient accumulation: 8
Effective batch size: 16
FP16: True
Gradient checkpointing: True

Configuration saved to: /content/icelandic_asr/training_config.json

STEP 3 COMPLETE


## Step 4 — Persistent Storage and Experiment Tracking

### Objective
Configure persistent Google Drive storage for Whisper Small fine-tuning.

### Storage Structure
- `configs/`: Training configurations and environment information.
- `checkpoints/`: Model checkpoints during training.
- `logs/`: Training and validation logs.
- `results/`: Training graphs, metrics, and predictions.
- `final_model/`: Best fine-tuned Whisper Small model.

### Requirements
1. Mount Google Drive.
2. Create the project directories.
3. Back up the training configuration.
4. Verify that files can be written and read.
5. Preserve all important training outputs across Colab sessions.

### Note
Checkpoints will be saved directly to Google Drive to prevent
data loss if the Colab runtime disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

print("=" * 60)
print("STEP 4 — PERSISTENT STORAGE SETUP")
print("=" * 60)

# Mount Google Drive
drive.mount("/content/drive")

# Main project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning"
)

# Define project folders
DIRECTORIES = {
    "configs": PROJECT_DIR / "configs",
    "checkpoints": PROJECT_DIR / "checkpoints",
    "logs": PROJECT_DIR / "logs",
    "results": PROJECT_DIR / "results",
    "final_model": PROJECT_DIR / "final_model"
}

# Create directories
for name, path in DIRECTORIES.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"Created/verified: {name}")

# Back up training configuration
source_config = Path(
    "/content/icelandic_asr/training_config.json"
)

destination_config = (
    DIRECTORIES["configs"] / "training_config.json"
)

assert source_config.exists(), (
    "Training configuration not found. "
    "Please rerun Step 3."
)

shutil.copy2(source_config, destination_config)

# Verify file integrity
with open(destination_config, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

assert saved_config["model_id"] == "openai/whisper-small"
assert saved_config["seed"] == 42

# Test Drive write/read access
test_file = PROJECT_DIR / "storage_test.txt"
test_file.write_text("Storage verification successful.", encoding="utf-8")

assert test_file.read_text(
    encoding="utf-8"
) == "Storage verification successful."

test_file.unlink()

print("\n" + "=" * 60)
print("STORAGE VERIFICATION")
print("=" * 60)

print(f"Project directory: {PROJECT_DIR}")
print(f"Configuration backup: {destination_config}")
print("Google Drive write/read test: PASSED")

print("\nSTEP 4 COMPLETE")

STEP 4 — PERSISTENT STORAGE SETUP
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created/verified: configs
Created/verified: checkpoints
Created/verified: logs
Created/verified: results
Created/verified: final_model

STORAGE VERIFICATION
Project directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning
Configuration backup: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/configs/training_config.json
Google Drive write/read test: PASSED

STEP 4 COMPLETE


## Step 5 — Whisper Fine-Tuning Pipeline

### Objective
Prepare the training and evaluation components for
Whisper Small Multilingual.

### Components
1. WER and CER evaluation metrics.
2. Whisper data collator.
3. Training arguments and optimization settings.
4. Checkpoint selection and experiment logging.

### Evaluation Strategy
- Training loss measures model optimization.
- Validation loss measures performance on unseen validation data.
- Validation WER measures word recognition errors.
- Validation CER measures character recognition errors.
- The best checkpoint will be selected using validation WER.

### Important
The test dataset will not be used during training.

Final statistical evaluation will be performed by Person 4.

In [ ]:
import evaluate
import numpy as np

print("=" * 60)
print("STEP 5A — EVALUATION METRICS")
print("=" * 60)

# Load evaluation metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("\nWER metric loaded successfully.")
print("CER metric loaded successfully.")

# Test the metrics using sample sentences
references = [
    "hello world",
    "this is a test"
]

predictions = [
    "hello world",
    "this is test"
]

wer = wer_metric.compute(
    predictions=predictions,
    references=references
)

cer = cer_metric.compute(
    predictions=predictions,
    references=references
)

print("\nMETRIC VALIDATION")
print("-" * 60)

print(f"WER: {wer:.4f} ({wer * 100:.2f}%)")
print(f"CER: {cer:.4f} ({cer * 100:.2f}%)")

# Verify expected WER:
# 1 deletion across 6 reference words
expected_wer = 1 / 6

assert np.isclose(wer, expected_wer), (
    f"Unexpected WER: {wer}"
)

assert 0 <= cer <= 1, "Unexpected CER value."

print("\nMetric validation: PASSED")
print("STEP 5A COMPLETE")

STEP 5A — EVALUATION METRICS



WER metric loaded successfully.
CER metric loaded successfully.

METRIC VALIDATION
------------------------------------------------------------
WER: 0.1667 (16.67%)
CER: 0.0800 (8.00%)

Metric validation: PASSED
STEP 5A COMPLETE


### Step 5B — Whisper Data Collator

**Objective:** Prepare audio features and tokenized transcripts for training.

**Data Collator Responsibilities:**
1. Combine audio features into training batches.
2. Pad variable-length transcript sequences.
3. Replace label padding with -100 to exclude it from loss calculations.
4. Handle Whisper's decoder start token correctly.
5. Produce tensors compatible with the Whisper training pipeline.

**Expected Output:**
A reusable data collator for training and validation.

**Note:** The collator will be tested with Person 1's dataset before training.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from transformers import WhisperProcessor

print("=" * 60)
print("STEP 5B — WHISPER DATA COLLATOR")
print("=" * 60)

# Reload processor after runtime restart
MODEL_ID = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="icelandic",
    task="transcribe"
)


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:

    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        # Prepare audio features
        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # Prepare transcript labels
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # Ignore padding tokens during loss calculation
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        # Remove decoder start token if already present
        decoder_start_token_id = (
            self.processor.tokenizer.convert_tokens_to_ids(
                "<|startoftranscript|>"
            )
        )

        if (
            labels.shape[1] > 0
            and (labels[:, 0] == decoder_start_token_id).all().item()
        ):
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


# Initialize data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

print("\nProcessor loaded successfully.")
print("Data collator initialized successfully.")

print("\nConfiguration:")
print("Model:", MODEL_ID)
print("Language: Icelandic")
print("Task: Transcription")
print("Padding labels masked with: -100")

print("\nSTEP 5B COMPLETE")

STEP 5B — WHISPER DATA COLLATOR

Processor loaded successfully.
Data collator initialized successfully.

Configuration:
Model: openai/whisper-small
Language: Icelandic
Task: Transcription
Padding labels masked with: -100

STEP 5B COMPLETE


### Step 5C — Training Arguments and Checkpoint Configuration

**Objective:** Configure the Hugging Face Seq2SeqTrainer for
Whisper Small fine-tuning.

**Training Configuration:**
- Learning rate: 1e-5
- Epochs: 3
- Training batch size: 2
- Gradient accumulation: 8
- Effective batch size: 16
- Precision: FP16
- Gradient checkpointing: Enabled
- Validation: Every epoch
- Checkpoint saving: Every epoch
- Best checkpoint: Lowest validation WER

**Experiment Tracking:**
- Record training loss and validation loss.
- Record validation WER and CER.
- Record training time and GPU memory usage.
- Save training logs and model checkpoints.

**Important:**
- Use validation data for checkpoint selection.
- Do not evaluate on the test set during training.
- Verify compatibility with the installed Transformers version.

In [ ]:
import json
import inspect
import torch

from pathlib import Path
from transformers import Seq2SeqTrainingArguments

print("=" * 60)
print("STEP 5C — TRAINING ARGUMENTS")
print("=" * 60)

# Reload configuration from Google Drive
PROJECT_DIR = Path(
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning"
)

config_path = PROJECT_DIR / "configs" / "training_config.json"

assert config_path.exists(), "Training configuration not found."

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

# Check supported arguments in installed Transformers
supported_args = inspect.signature(
    Seq2SeqTrainingArguments
).parameters

# Handle differences between Transformers versions
if "eval_strategy" in supported_args:
    eval_key = "eval_strategy"
elif "evaluation_strategy" in supported_args:
    eval_key = "evaluation_strategy"
else:
    raise RuntimeError(
        "No supported evaluation strategy argument found."
    )

print(f"\nEvaluation argument: {eval_key}")

# Training arguments
args_dict = {
    "output_dir": str(PROJECT_DIR / "checkpoints"),

    "learning_rate": config["learning_rate"],
    "num_train_epochs": config["num_train_epochs"],

    "per_device_train_batch_size":
        config["per_device_train_batch_size"],

    "per_device_eval_batch_size":
        config["per_device_eval_batch_size"],

    "gradient_accumulation_steps":
        config["gradient_accumulation_steps"],

    "weight_decay": config["weight_decay"],
    "warmup_steps": 0,
    "lr_scheduler_type": config["lr_scheduler_type"],

    "fp16": config["fp16"],
    "gradient_checkpointing":
        config["gradient_checkpointing"],

    "logging_strategy": "steps",
    "logging_steps": config["logging_steps"],

    "save_strategy": config["save_strategy"],
    "save_total_limit": config["save_total_limit"],

    "predict_with_generate": True,
    "generation_max_length": 225,

    "load_best_model_at_end": True,
    "metric_for_best_model":
        config["metric_for_best_model"],

    "greater_is_better":
        config["greater_is_better"],

    "seed": config["seed"],

    "report_to": "none",

    "remove_unused_columns": False,
    "dataloader_num_workers": 0
}

args_dict[eval_key] = config["evaluation_strategy"]

# Verify argument compatibility
unsupported = [
    key for key in args_dict
    if key not in supported_args
]

if unsupported:
    raise RuntimeError(
        f"Unsupported training arguments: {unsupported}"
    )

# Initialize training arguments
training_args = Seq2SeqTrainingArguments(**args_dict)

print("\nTRAINING CONFIGURATION")
print("-" * 60)

print("Learning rate:", training_args.learning_rate)
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)

print("\nCHECKPOINT CONFIGURATION")
print("-" * 60)

print("Output directory:", training_args.output_dir)
print("Save strategy:", training_args.save_strategy)
print("Best model metric:", training_args.metric_for_best_model)
print("Load best model:", training_args.load_best_model_at_end)

print("\nSTEP 5C COMPLETE")

STEP 5C — TRAINING ARGUMENTS

Evaluation argument: eval_strategy

TRAINING CONFIGURATION
------------------------------------------------------------
Learning rate: 1e-05
Epochs: 3
Batch size: 2
Gradient accumulation: 8
FP16: True
Gradient checkpointing: True

CHECKPOINT CONFIGURATION
------------------------------------------------------------
Output directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/checkpoints
Save strategy: SaveStrategy.EPOCH
Best model metric: wer
Load best model: True

STEP 5C COMPLETE


### Step 5D — Training Configuration Validation

**Objective:** Validate the training configuration before initializing the trainer.

**Checks:**
- Confirm the training and evaluation strategies match.
- Confirm the best checkpoint is selected using validation WER.
- Confirm FP16 is enabled and CUDA is available.
- Confirm the effective batch size.
- Identify settings that must be finalized after receiving the dataset.

**Expected result:** All configuration checks pass.

## Step 6 — Whisper Trainer Preparation

### Step 6A — Model Initialization and GPU Verification

**Objective:** Reload the pretrained Whisper Small multilingual model and verify GPU compatibility.

**Model:** `openai/whisper-small`

**Language:** Icelandic

**Task:** Automatic Speech Recognition (Transcription)

**Checks:**
- Load the pretrained model.
- Configure Icelandic transcription.
- Verify model parameters.
- Verify GPU compatibility.
- Record GPU memory usage.

**Expected Output:** Whisper Small is initialized successfully and ready for trainer integration.

**Note:** No training or test-set evaluation occurs in this step.

# Step 7 — Dataset Integration

## Step 7A — Repository Setup

Retrieve Person 1's prepared Icelandic ASR dataset and inspect
the repository structure before loading the audio and transcripts.

In [ ]:
from pathlib import Path
import subprocess
import os

print("=" * 60)
print("STEP 7A — DATASET REPOSITORY SETUP")
print("=" * 60)

repo_url = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
repo_path = Path("/content/icelandic-asr-project3")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_path)],
        check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    )
else:
    print("Repository already exists.")

print("\nREPOSITORY CONTENTS")
print("-" * 60)

for item in sorted(repo_path.iterdir()):
    print(f"{'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

print("\nSTEP 7A COMPLETE")

STEP 7A — DATASET REPOSITORY SETUP

REPOSITORY CONTENTS
------------------------------------------------------------
[DIR] .git
[FILE] .gitattributes
[FILE] .gitignore
[FILE] BASELINES.md
[FILE] DATA_STORAGE.md
[FILE] LICENSE
[FILE] README.md
[FILE] audit.py
[DIR] contributions
[DIR] data
[DIR] output
[FILE] prepare.py
[FILE] preprocess.py
[FILE] requirements-windows-py314.lock.txt
[FILE] requirements.txt
[DIR] review
[FILE] subset.py
[DIR] tests
[FILE] verify.py
[FILE] verify_handoff.py

STEP 7A COMPLETE


## Step 7B — Dataset File Verification

Verify the training, validation, and test manifests.

Check that the audio recordings are available and identify
whether Git LFS files were downloaded correctly.

In [ ]:
from pathlib import Path
import subprocess

print("=" * 60)
print("STEP 7B — DATASET FILE VERIFICATION")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")

# Inspect the output directory
print("\nOUTPUT DIRECTORY")
print("-" * 60)

output_dir = repo / "output"

for item in sorted(output_dir.rglob("*")):
    if item.is_file():
        print(f"{item.relative_to(repo)} — {item.stat().st_size:,} bytes")

# Inspect the data directory
print("\nDATA DIRECTORY")
print("-" * 60)

data_dir = repo / "data"
audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

audio_files = [
    p for p in data_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in audio_extensions
]

print(f"Audio files found: {len(audio_files):,}")

if audio_files:
    print("\nSample audio files:")
    for file in audio_files[:5]:
        print(f"  {file.relative_to(repo)} — {file.stat().st_size:,} bytes")

# Verify expected manifests
print("\nMANIFEST VERIFICATION")
print("-" * 60)

for name in ["train.jsonl", "dev.jsonl", "test.jsonl"]:
    path = output_dir / name

    if path.exists():
        with path.open(encoding="utf-8") as f:
            count = sum(1 for line in f if line.strip())

        print(f"{name}: FOUND — {count:,} entries")
    else:
        print(f"{name}: NOT FOUND")

# Check Git LFS availability
print("\nGIT LFS STATUS")
print("-" * 60)

result = subprocess.run(
    ["git", "lfs", "version"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout.strip())
else:
    print("Git LFS is not installed.")

print("\nSTEP 7B COMPLETE")

STEP 7B — DATASET FILE VERIFICATION

OUTPUT DIRECTORY
------------------------------------------------------------
output/audio_audit.jsonl.gz — 10,327,189 bytes
output/ctc_vocab.json — 429 bytes
output/dev.jsonl — 657,876 bytes
output/evaluation_references.jsonl — 564,522 bytes
output/evaluation_references.tsv — 411,047 bytes
output/exclusions.tsv — 10,477 bytes
output/flagged_dev_test_review.tsv — 102,771 bytes
output/focused_review.tsv — 10,139 bytes
output/handoff_verification.json — 2,479 bytes
output/lfs_verification.json — 1,698 bytes
output/listening_queue.tsv — 2,001,220 bytes
output/prompt_overlap.tsv — 748 bytes
output/quality_report.md — 5,273 bytes
output/quality_summary.json — 8,906 bytes
output/raw_subset_cleaning.json — 777 bytes
output/raw_subset_inventory.tsv — 3,089,955 bytes
output/raw_subset_removals.tsv — 827 bytes
output/split_integrity.json — 751 bytes
output/test.jsonl — 969,000 bytes
output/train.jsonl — 5,200,398 bytes
output/verification.json — 2,278 bytes



## Step 7C — Git LFS Audio Retrieval

Install Git LFS and retrieve the actual Icelandic audio recordings.

Git LFS pointer files must be replaced with real audio data
before Whisper preprocessing and training.

In [ ]:
import subprocess
from pathlib import Path

print("=" * 60)
print("STEP 7C — GIT LFS AUDIO RETRIEVAL")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")

# Install Git LFS
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "git-lfs"],
    check=True
)

# Initialize Git LFS
subprocess.run(
    ["git", "lfs", "install"],
    cwd=repo,
    check=True
)

# Download tracked audio files
print("\nDownloading Git LFS audio files...")

subprocess.run(
    ["git", "lfs", "pull"],
    cwd=repo,
    check=True
)

print("\nVERIFY AUDIO FILES")
print("-" * 60)

audio_files = list((repo / "data").rglob("*.flac"))

if not audio_files:
    raise FileNotFoundError("No FLAC audio files found.")

sizes = [p.stat().st_size for p in audio_files]

print(f"FLAC files: {len(audio_files):,}")
print(f"Smallest file: {min(sizes):,} bytes")
print(f"Largest file: {max(sizes):,} bytes")

pointer_files = sum(size < 200 for size in sizes)

print(f"Possible LFS pointers remaining: {pointer_files:,}")

print("\nSTEP 7C COMPLETE")

STEP 7C — GIT LFS AUDIO RETRIEVAL


VERIFY AUDIO FILES
------------------------------------------------------------
FLAC files: 30,993
Smallest file: 15,901 bytes
Largest file: 221,849 bytes
Possible LFS pointers remaining: 0

STEP 7C COMPLETE


## Step 7D — Manifest and Audio Path Verification

Inspect the dataset manifests and verify that their audio
references correspond to existing recordings.

This step does not modify the dataset.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("STEP 7D — MANIFEST INSPECTION")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
output_dir = repo / "output"

for split in ["train", "dev", "test"]:

    manifest = output_dir / f"{split}.jsonl"

    with manifest.open(encoding="utf-8") as f:
        first_record = json.loads(f.readline())

    print(f"\n{split.upper()} SPLIT")
    print("-" * 60)

    print("Available fields:")
    print(list(first_record.keys()))

    print("\nFirst record:")

    for key, value in first_record.items():
        display_value = str(value)

        if len(display_value) > 200:
            display_value = display_value[:200] + "..."

        print(f"{key}: {display_value}")

print("\nSTEP 7D COMPLETE")

STEP 7D — MANIFEST INSPECTION

TRAIN SPLIT
------------------------------------------------------------
Available fields:
['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags']

First record:
id: 0072202
split: train
speaker_session_id: 007478
audio_path: train/007478/007478-0072202.flac
sample_rate: 16000
num_samples: 45120
duration_sec: 2.82
text_raw: Ég vil vita meira.
text_source_normalized: ég vil vita meira
text: ég vil vita meira
pcm_sha256: 9a8964624c97480c5228113fe1ae113793a241994fb5edd8fa203061f161accb
flags: []

DEV SPLIT
------------------------------------------------------------
Available fields:
['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags']

First record:
id: 0034220
split: dev
speaker_session_id: 004485
audio_path: dev/004485/004485-0034220.

## Step 7E — Audio Path Integrity Check

Verify that all training, validation, and test recordings
exist and are accessible.

This step checks file paths without modifying the dataset.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("STEP 7E — AUDIO PATH INTEGRITY")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
output_dir = repo / "output"

# Person 1's prepared audio directory
audio_root = repo / "data" / "cleaned" / "samromur-21.05-20h-v1"

print(f"\nAudio directory: {audio_root}")
print(f"Directory exists: {audio_root.exists()}")

missing = []

for split in ["train", "dev", "test"]:
    manifest = output_dir / f"{split}.jsonl"
    total = 0
    found = 0

    with manifest.open(encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            total += 1

            audio_file = audio_root / record["audio_path"]

            if audio_file.is_file() and audio_file.stat().st_size > 200:
                found += 1
            else:
                missing.append(str(audio_file))

    print(f"\n{split.upper()}")
    print(f"Expected: {total:,}")
    print(f"Available: {found:,}")
    print(f"Missing: {total - found:,}")

print("\n" + "-" * 60)

if not missing:
    print("ALL AUDIO PATHS VERIFIED")
else:
    print(f"Missing or invalid recordings: {len(missing):,}")
    print("First five:")
    for path in missing[:5]:
        print(path)

print("\nSTEP 7E COMPLETE")

STEP 7E — AUDIO PATH INTEGRITY

Audio directory: /content/icelandic-asr-project3/data/cleaned/samromur-21.05-20h-v1
Directory exists: True

TRAIN
Expected: 10,488
Available: 10,488
Missing: 0

DEV
Expected: 1,314
Available: 1,314
Missing: 0

TEST
Expected: 1,884
Available: 1,884
Missing: 0

------------------------------------------------------------
ALL AUDIO PATHS VERIFIED

STEP 7E COMPLETE


# Step 8 — Whisper Dataset Preparation

## Step 8A — Load Dataset Manifests

Load the prepared Icelandic ASR training, validation, and test
manifests into Hugging Face Datasets.

Preserve the original dataset splits and normalized transcriptions.

In [ ]:
from pathlib import Path
from datasets import load_dataset, DatasetDict

print("=" * 60)
print("STEP 8A — LOAD ICELANDIC DATASET")
print("=" * 60)

repo = Path("/content/icelandic-asr-project3")
output_dir = repo / "output"

# Load Person 1's prepared manifests
dataset = load_dataset(
    "json",
    data_files={
        "train": str(output_dir / "train.jsonl"),
        "validation": str(output_dir / "dev.jsonl"),
        "test": str(output_dir / "test.jsonl"),
    }
)

# Convert to DatasetDict
dataset = DatasetDict(dataset)

print("\nDATASET STRUCTURE")
print("-" * 60)
print(dataset)

print("\nSPLIT SIZES")
print("-" * 60)

for split in dataset:
    print(f"{split}: {len(dataset[split]):,} recordings")

print("\nSAMPLE TRANSCRIPTION")
print("-" * 60)

sample = dataset["train"][0]

print("ID:", sample["id"])
print("Audio path:", sample["audio_path"])
print("Transcription:", sample["text"])
print("Duration:", sample["duration_sec"], "seconds")
print("Sample rate:", sample["sample_rate"], "Hz")

# Verify expected split sizes
assert len(dataset["train"]) == 10488
assert len(dataset["validation"]) == 1314
assert len(dataset["test"]) == 1884

print("\nDATASET VALIDATION PASSED")
print("STEP 8A COMPLETE")

STEP 8A — LOAD ICELANDIC DATASET


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


DATASET STRUCTURE
------------------------------------------------------------
DatasetDict({
    train: Dataset({
        features: ['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags'],
        num_rows: 10488
    })
    validation: Dataset({
        features: ['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags'],
        num_rows: 1314
    })
    test: Dataset({
        features: ['id', 'split', 'speaker_session_id', 'audio_path', 'sample_rate', 'num_samples', 'duration_sec', 'text_raw', 'text_source_normalized', 'text', 'pcm_sha256', 'flags'],
        num_rows: 1884
    })
})

SPLIT SIZES
------------------------------------------------------------
train: 10,488 recordings
validation: 1,314 recordings
test: 1,884 recordings

SAMPLE TRANSCRIPTION
--------------

## Step 8B — Audio Loading and Feature Extraction Test

Load one Icelandic recording and convert it into Whisper
input features and tokenized transcription labels.

Verify audio decoding, sampling rate, and tensor dimensions
before processing the full dataset.

In [ ]:
import numpy as np
import soundfile as sf
from pathlib import Path

print("=" * 60)
print("STEP 8B — WHISPER FEATURE EXTRACTION TEST")
print("=" * 60)

audio_root = (
    Path("/content/icelandic-asr-project3")
    / "data"
    / "cleaned"
    / "samromur-21.05-20h-v1"
)

# Select one training recording
sample = dataset["train"][0]

audio_path = audio_root / sample["audio_path"]

# Load audio
audio, sample_rate = sf.read(
    str(audio_path),
    dtype="float32"
)

print("\nAUDIO INFORMATION")
print("-" * 60)
print("File:", audio_path.name)
print("Sample rate:", sample_rate)
print("Duration:", round(len(audio) / sample_rate, 2), "seconds")
print("Audio shape:", audio.shape)

assert sample_rate == 16000, "Unexpected sample rate"
assert audio.ndim == 1, "Expected mono audio"
assert np.isfinite(audio).all(), "Audio contains invalid values"

# Extract Whisper input features
input_features = processor.feature_extractor(
    audio,
    sampling_rate=sample_rate,
    return_tensors="pt"
).input_features

# Tokenize transcription
labels = processor.tokenizer(
    sample["text"]
).input_ids

print("\nWHISPER FEATURES")
print("-" * 60)
print("Input feature shape:", tuple(input_features.shape))
print("Number of label tokens:", len(labels))
print("Transcription:", sample["text"])

# Validate features
assert input_features.shape[0] == 1
assert input_features.shape[1] == 80
assert len(labels) > 0

print("\nFEATURE EXTRACTION VALIDATION PASSED")
print("STEP 8B COMPLETE")

STEP 8B — WHISPER FEATURE EXTRACTION TEST

AUDIO INFORMATION
------------------------------------------------------------
File: 007478-0072202.flac
Sample rate: 16000
Duration: 2.82 seconds
Audio shape: (45120,)

WHISPER FEATURES
------------------------------------------------------------
Input feature shape: (1, 80, 3000)
Number of label tokens: 10
Transcription: ég vil vita meira

FEATURE EXTRACTION VALIDATION PASSED
STEP 8B COMPLETE


## Step 8C — Training and Validation Preprocessing

Convert the prepared Icelandic audio recordings into Whisper
log-Mel features and tokenize the corresponding transcriptions.

Preserve the original dataset splits and leave the test set
untouched until final evaluation.

In [ ]:
import numpy as np
import soundfile as sf
from pathlib import Path

print("=" * 60)
print("STEP 8C — WHISPER DATASET PREPROCESSING")
print("=" * 60)

audio_root = (
    Path("/content/icelandic-asr-project3")
    / "data"
    / "cleaned"
    / "samromur-21.05-20h-v1"
)

def prepare_whisper_sample(example):

    audio_path = audio_root / example["audio_path"]

    audio, sample_rate = sf.read(
        str(audio_path),
        dtype="float32"
    )

    if sample_rate != 16000:
        raise ValueError(
            f"Unexpected sample rate: {sample_rate}"
        )

    if audio.ndim != 1:
        raise ValueError(
            f"Expected mono audio: {audio_path}"
        )

    # Extract Whisper log-Mel features
    features = processor.feature_extractor(
        audio,
        sampling_rate=16000
    ).input_features[0]

    # Tokenize transcription
    labels = processor.tokenizer(
        example["text"]
    ).input_ids

    return {
        "input_features": features,
        "labels": labels
    }


print("\nPreparing training dataset...")

train_dataset = dataset["train"].map(
    prepare_whisper_sample,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
    desc="Preparing training audio"
)

print("\nPreparing validation dataset...")

eval_dataset = dataset["validation"].map(
    prepare_whisper_sample,
    remove_columns=dataset["validation"].column_names,
    num_proc=1,
    desc="Preparing validation audio"
)

print("\nPREPROCESSING RESULTS")
print("-" * 60)

print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(eval_dataset):,}")

print("\nTraining features:")
print(train_dataset.features)

assert len(train_dataset) == 10488
assert len(eval_dataset) == 1314

print("\nSTEP 8C COMPLETE")

STEP 8C — WHISPER DATASET PREPROCESSING

Preparing training dataset...


Preparing training audio (num_proc=1):   0%|          | 0/10488 [00:00<?, ? examples/s]


Preparing validation dataset...


Preparing validation audio (num_proc=1):   0%|          | 0/1314 [00:00<?, ? examples/s]


PREPROCESSING RESULTS
------------------------------------------------------------
Training samples: 10,488
Validation samples: 1,314

Training features:
{'input_features': List(List(Value('float32'))), 'labels': List(Value('int64'))}

STEP 8C COMPLETE


# Step 9 — Final Training Configuration

## Step 9A — Calculate Training Steps and Warmup

Calculate the total number of optimizer updates and set
the learning-rate warmup to 5% of the training schedule.

Verify the training configuration before initializing
the Hugging Face Trainer.

In [ ]:
import math

print("=" * 60)
print("STEP 9A — TRAINING SCHEDULE")
print("=" * 60)

# Dataset
num_train_samples = len(train_dataset)

# Training configuration
batch_size = training_args.per_device_train_batch_size
gradient_accumulation = training_args.gradient_accumulation_steps
num_epochs = training_args.num_train_epochs

# Single GPU training
num_gpus = 1

effective_batch_size = (
    batch_size
    * gradient_accumulation
    * num_gpus
)

# Calculate optimizer steps
batches_per_epoch = math.ceil(
    num_train_samples / (batch_size * num_gpus)
)

steps_per_epoch = math.ceil(
    batches_per_epoch / gradient_accumulation
)

total_steps = math.ceil(
    steps_per_epoch * num_epochs
)

# 5% warmup
warmup_steps = math.ceil(total_steps * 0.05)

# Update training arguments
training_args.warmup_steps = warmup_steps

print("\nTRAINING CONFIGURATION")
print("-" * 60)

print(f"Training samples: {num_train_samples:,}")
print(f"Epochs: {num_epochs}")
print(f"Batch size per GPU: {batch_size}")
print(f"Gradient accumulation: {gradient_accumulation}")
print(f"Effective batch size: {effective_batch_size}")

print("\nOPTIMIZER SCHEDULE")
print("-" * 60)

print(f"Batches per epoch: {batches_per_epoch:,}")
print(f"Optimizer steps per epoch: {steps_per_epoch:,}")
print(f"Total optimizer steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

print("\nTRAINING ARGUMENTS")
print("-" * 60)

print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)
print("Evaluation strategy:", training_args.eval_strategy)
print("Save strategy:", training_args.save_strategy)
print("Warmup steps:", training_args.warmup_steps)

assert training_args.warmup_steps > 0
assert training_args.fp16 is True
assert training_args.load_best_model_at_end is True

print("\nSTEP 9A COMPLETE")

STEP 9A — TRAINING SCHEDULE

TRAINING CONFIGURATION
------------------------------------------------------------
Training samples: 10,488
Epochs: 3
Batch size per GPU: 2
Gradient accumulation: 8
Effective batch size: 16

OPTIMIZER SCHEDULE
------------------------------------------------------------
Batches per epoch: 5,244
Optimizer steps per epoch: 656
Total optimizer steps: 1,968
Warmup steps: 99

TRAINING ARGUMENTS
------------------------------------------------------------
Learning rate: 1e-05
FP16: True
Gradient checkpointing: True
Evaluation strategy: IntervalStrategy.EPOCH
Save strategy: SaveStrategy.EPOCH
Warmup steps: 99

STEP 9A COMPLETE


## Step 9B — WER and CER Evaluation

Configure Word Error Rate (WER) and Character Error Rate (CER)
for evaluating Whisper Small on the Icelandic validation set.

Decode model predictions and reference transcriptions,
then calculate both metrics.

Use WER to select the best training checkpoint.

In [ ]:
import numpy as np
import evaluate

print("=" * 60)
print("STEP 9B — EVALUATION METRICS")
print("=" * 60)

# Load evaluation metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):

    # Predicted token IDs
    pred_ids = pred.predictions

    # Handle models returning additional outputs
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # Reference token IDs
    label_ids = pred.label_ids.copy()

    # Replace ignored labels with padding token
    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    # Decode predictions
    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    # Decode references
    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    # Normalize whitespace
    pred_str = [
        " ".join(text.strip().split())
        for text in pred_str
    ]

    label_str = [
        " ".join(text.strip().split())
        for text in label_str
    ]

    # Calculate metrics
    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }


# Test evaluation function
class DummyPrediction:
    def __init__(self, predictions, label_ids):
        self.predictions = predictions
        self.label_ids = label_ids


reference = "ég vil vita meira"
prediction = "ég vil vita meira"

reference_ids = processor.tokenizer(
    reference
).input_ids

prediction_ids = processor.tokenizer(
    prediction
).input_ids

# Pad sequences for testing
max_length = max(
    len(reference_ids),
    len(prediction_ids)
)

pad_id = processor.tokenizer.pad_token_id

reference_ids += [pad_id] * (
    max_length - len(reference_ids)
)

prediction_ids += [pad_id] * (
    max_length - len(prediction_ids)
)

dummy = DummyPrediction(
    predictions=np.array([prediction_ids]),
    label_ids=np.array([reference_ids])
)

results = compute_metrics(dummy)

print("\nEVALUATION TEST")
print("-" * 60)

print("Reference:", reference)
print("Prediction:", prediction)

print(f"WER: {results['wer']:.4f}")
print(f"CER: {results['cer']:.4f}")

assert results["wer"] == 0.0
assert results["cer"] == 0.0

print("\nEVALUATION METRICS VERIFIED")
print("STEP 9B COMPLETE")

STEP 9B — EVALUATION METRICS


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



EVALUATION TEST
------------------------------------------------------------
Reference: ég vil vita meira
Prediction: ég vil vita meira
WER: 0.0000
CER: 0.0000

EVALUATION METRICS VERIFIED
STEP 9B COMPLETE


## Step 9C — Initialize Whisper Trainer

Initialize the Hugging Face Seq2SeqTrainer using the
preprocessed Icelandic dataset, Whisper Small model,
training configuration, data collator, and evaluation metrics.

Verify the training environment before fine-tuning.

In [ ]:
import torch
from transformers import Seq2SeqTrainer

print("=" * 60)
print("STEP 9C — WHISPER TRAINER INITIALIZATION")
print("=" * 60)

# Disable BPE token cleanup
processor.tokenizer.clean_up_tokenization_spaces = False

# Ensure generation-based evaluation is enabled
training_args.predict_with_generate = True

# Configure Whisper generation
model.generation_config.language = "icelandic"
model.generation_config.task = "transcribe"

# Disable cache during gradient checkpointing
model.config.use_cache = False

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\nTRAINER CONFIGURATION")
print("-" * 60)

print("Model:", model.config.model_type)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print("Epochs:", trainer.args.num_train_epochs)
print("Batch size:", trainer.args.per_device_train_batch_size)
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("Learning rate:", trainer.args.learning_rate)
print("Warmup steps:", trainer.args.warmup_steps)
print("FP16:", trainer.args.fp16)
print("Gradient checkpointing:", trainer.args.gradient_checkpointing)
print("Generate predictions:", trainer.args.predict_with_generate)
print("Checkpoint directory:", trainer.args.output_dir)

print("\nGPU")
print("-" * 60)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

assert len(trainer.train_dataset) == 10488
assert len(trainer.eval_dataset) == 1314
assert trainer.args.warmup_steps == 99
assert trainer.args.predict_with_generate is True
assert torch.cuda.is_available()

print("\nTRAINER INITIALIZATION PASSED")
print("STEP 9C COMPLETE")

STEP 9C — WHISPER TRAINER INITIALIZATION

TRAINER CONFIGURATION
------------------------------------------------------------
Model: whisper
Training samples: 10488
Validation samples: 1314
Epochs: 3
Batch size: 2
Gradient accumulation: 8
Learning rate: 1e-05
Warmup steps: 99
FP16: True
Gradient checkpointing: True
Generate predictions: True
Checkpoint directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/checkpoints

GPU
------------------------------------------------------------
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

TRAINER INITIALIZATION PASSED
STEP 9C COMPLETE


## Step 9D — GPU Training Compatibility Test

Test one forward and backward pass using the actual
Icelandic training dataset.

Verify GPU memory usage, loss calculation, and gradient
computation before starting full fine-tuning.

No optimizer step is performed during this test.

# Step 10 — Whisper Small Fine-Tuning

## Step 10A — Checkpoint and Recovery Configuration

Configure periodic checkpoint saving and validation
during fine-tuning.

Save checkpoints to Google Drive so training can resume
after a Colab disconnection.

Retain the latest checkpoints and the best model
according to validation WER.

In [ ]:
from pathlib import Path
from transformers import IntervalStrategy

print("=" * 60)
print("STEP 10A — CHECKPOINT RECOVERY CONFIGURATION")
print("=" * 60)

# Google Drive checkpoint directory
checkpoint_dir = Path(
    "/content/drive/MyDrive/"
    "NLP_Project3_Whisper_FineTuning/checkpoints"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Configure periodic evaluation and saving
training_args.eval_strategy = IntervalStrategy.STEPS
training_args.save_strategy = IntervalStrategy.STEPS

training_args.eval_steps = 200
training_args.save_steps = 200

# Retain a limited number of checkpoints
training_args.save_total_limit = 2

# Select best model using validation WER
training_args.load_best_model_at_end = True
training_args.metric_for_best_model = "wer"
training_args.greater_is_better = False

# Keep generation-based evaluation enabled
training_args.predict_with_generate = True

# Reinitialize Trainer with updated arguments
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\nCHECKPOINT CONFIGURATION")
print("-" * 60)

print("Checkpoint directory:", trainer.args.output_dir)
print("Save strategy:", trainer.args.save_strategy)
print("Save steps:", trainer.args.save_steps)
print("Evaluation strategy:", trainer.args.eval_strategy)
print("Evaluation steps:", trainer.args.eval_steps)
print("Checkpoint retention:", trainer.args.save_total_limit)
print("Best model metric:", trainer.args.metric_for_best_model)
print("Lower metric is better:", not trainer.args.greater_is_better)

print("\nEXISTING CHECKPOINTS")
print("-" * 60)

checkpoints = sorted(
    checkpoint_dir.glob("checkpoint-*")
)

if checkpoints:
    for checkpoint in checkpoints:
        print(checkpoint.name)
else:
    print("No previous checkpoints found.")

# Validation
assert trainer.args.save_steps == 200
assert trainer.args.eval_steps == 200
assert trainer.args.save_total_limit == 2
assert trainer.args.load_best_model_at_end
assert trainer.args.metric_for_best_model == "wer"

print("\nCHECKPOINT CONFIGURATION VERIFIED")
print("STEP 10A COMPLETE")

STEP 10A — CHECKPOINT RECOVERY CONFIGURATION

CHECKPOINT CONFIGURATION
------------------------------------------------------------
Checkpoint directory: /content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/checkpoints
Save strategy: IntervalStrategy.STEPS
Save steps: 200
Evaluation strategy: IntervalStrategy.STEPS
Evaluation steps: 200
Checkpoint retention: 2
Best model metric: wer
Lower metric is better: True

EXISTING CHECKPOINTS
------------------------------------------------------------
No previous checkpoints found.

CHECKPOINT CONFIGURATION VERIFIED
STEP 10A COMPLETE


> **Archived training run:** The output below records the original completed 1,968-step training run (5.20 hours). **Do not rerun this cell** during model inspection; doing so may start or resume expensive training. The original best validation WER was 30.62% at step 1600, but those weights were not recovered.



## Step 10B — Whisper Small Fine-Tuning

Fine-tune multilingual Whisper Small on the Icelandic
Samrómur dataset.

Training configuration:
- 3 epochs
- Learning rate: 1e-5
- Effective batch size: 16
- FP16 mixed precision
- Gradient checkpointing
- Checkpoints every 200 optimizer steps
- Validation using WER and CER

Record training duration, GPU information, and final metrics.

In [ ]:
import time
import json
import torch
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

print("=" * 60)
print("STEP 10B — WHISPER SMALL FINE-TUNING")
print("=" * 60)

# Directories
project_dir = Path(
    "/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning"
)

checkpoint_dir = project_dir / "checkpoints"
results_dir = project_dir / "results"

results_dir.mkdir(parents=True, exist_ok=True)

# Verify GPU
assert torch.cuda.is_available(), "GPU unavailable!"

gpu_name = torch.cuda.get_device_name(0)

print("\nTRAINING INFORMATION")
print("-" * 60)

print("GPU:", gpu_name)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(eval_dataset))
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Effective batch size:",
      trainer.args.per_device_train_batch_size *
      trainer.args.gradient_accumulation_steps)

# Check for previous checkpoints
last_checkpoint = get_last_checkpoint(
    str(checkpoint_dir)
)

if last_checkpoint:
    print("\nExisting checkpoint found:")
    print(last_checkpoint)
    print("Training will resume from this checkpoint.")
else:
    print("\nNo checkpoint found.")
    print("Starting fine-tuning from the pretrained model.")

# Record starting time
start_time = time.time()

print("\nSTARTING TRAINING...")
print("=" * 60)

# Train or resume
train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

# Record training duration
training_duration = time.time() - start_time

print("\nTRAINING COMPLETED")
print("-" * 60)

print(
    f"Training duration: {training_duration / 3600:.2f} hours"
)

print("Final training loss:", train_result.training_loss)

# Save training metrics
training_results = {
    "model": "openai/whisper-small",
    "language": "Icelandic",
    "gpu": gpu_name,
    "training_samples": len(train_dataset),
    "validation_samples": len(eval_dataset),
    "epochs": trainer.args.num_train_epochs,
    "learning_rate": trainer.args.learning_rate,
    "effective_batch_size": (
        trainer.args.per_device_train_batch_size *
        trainer.args.gradient_accumulation_steps
    ),
    "training_duration_seconds": training_duration,
    "training_loss": train_result.training_loss,
    "global_step": trainer.state.global_step
}

results_path = results_dir / "training_results.json"

with open(results_path, "w") as f:
    json.dump(training_results, f, indent=4)

print("\nTraining results saved to:")
print(results_path)

print("\nSTEP 10B COMPLETE")

STEP 10B — WHISPER SMALL FINE-TUNING

TRAINING INFORMATION
------------------------------------------------------------
GPU: Tesla T4
Training samples: 10488
Validation samples: 1314
Epochs: 3
Learning rate: 1e-05
Effective batch size: 16

No checkpoint found.
Starting fine-tuning from the pretrained model.

STARTING TRAINING...


Step,Training Loss,Validation Loss,Wer,Cer
200,5.860062,0.734294,0.509447,0.159794
400,4.098164,0.568799,0.424523,0.122550
600,3.727689,0.494144,0.382061,0.108491
800,2.202216,0.450531,0.368893,0.108507
1000,2.112906,0.429902,0.345134,0.096969
1200,2.085521,0.403523,0.322519,0.087826
1400,1.055174,0.401283,0.315935,0.086731
1600,1.036301,0.394304,0.306202,0.084147


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (6 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



TRAINING COMPLETED
------------------------------------------------------------
Training duration: 5.20 hours
Final training loss: 2.9130900536126236

Training results saved to:
/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/results/training_results.json

STEP 10B COMPLETE


## Step 11 — Recovered Model and Inference

The original run finished 1,968 optimizer steps. The best recorded validation WER was **30.62% at step 1600**; its checkpoint is unavailable. The independently exported **step-1000 recovery model** has validation WER **34.51%** and CER **9.70%**. Do not attribute the step-1600 result to the exported weights. The model is already stored in Google Drive at `NLP_Project3_Whisper_FineTuning/recovered_model_step1000`.

This section loads that existing export and runs a **single validation-recording smoke test**, not a held-out test-set evaluation. It requires only one FLAC recording.


In [ ]:
from pathlib import Path
import json, subprocess, os
import torch, soundfile as sf
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

repo = Path("/content/icelandic-asr-project3")
model_dir = Path("/content/drive/MyDrive/NLP_Project3_Whisper_FineTuning/recovered_model_step1000")
assert model_dir.is_dir(), "Mount Google Drive using Step 4 first."
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/RanimAlyaseen/icelandic-asr-project3.git", str(repo)], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})

with open(repo / "output/dev.jsonl", encoding="utf-8") as f:
    sample = json.loads(f.readline())
relative_audio = "data/cleaned/samromur-21.05-20h-v1/" + sample["audio_path"]
audio_path = repo / relative_audio
if not audio_path.exists() or audio_path.stat().st_size < 200:
    subprocess.run(["git", "lfs", "pull", "--include", relative_audio, "--exclude", ""], cwd=repo, check=True)
assert audio_path.exists(), f"Audio missing: {audio_path}"
with open(audio_path, "rb") as f:
    assert f.read(4) == b"fLaC", "Audio is not a downloaded FLAC file."

processor = WhisperProcessor.from_pretrained(str(model_dir), language="icelandic", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(str(model_dir), dtype=torch.float32).to("cpu").eval()
audio, sr = sf.read(str(audio_path))
if audio.ndim > 1: audio = audio.mean(axis=1)
assert sr == 16000, f"Unexpected sample rate: {sr}"
inputs = processor(audio, sampling_rate=sr, return_tensors="pt", return_attention_mask=True)
with torch.inference_mode():
    ids = model.generate(inputs.input_features, attention_mask=inputs.attention_mask, language="icelandic", task="transcribe", max_new_tokens=128)
prediction = processor.batch_decode(ids, skip_special_tokens=True)[0]
print("REFERENCE:", sample["text"])
print("PREDICTION:", prediction)
print(f"Single-sample WER: {wer(sample['text'], prediction):.2%}")
print(f"Single-sample CER: {cer(sample['text'], prediction):.2%}")
print("Note: Single-sample metrics are not the final test-set scores.")
